# DAPT Score — Partial C-index Evaluation

Computes a partial DAPT score using **6 of 9 variables** (Yeh et al., *JAMA* 2016).

**Missing variables:** stent diameter <3mm (+1 pt) and vein graft stent (+2 pts) — not captured in structured EHR records. Partial score represents a lower bound of true DAPT score performance.

**Important Limitations:**
- DAPT score designed for **post-12-month** decisions — entirely outside this study's 0-12 month prediction window
- Test set uses random_state=44 with stratification on ischemic label, consistent with main model evaluation

## 1. Imports & Paths

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from lifelines.utils import concordance_index

RANDOM_STATE = 44
N_BOOTSTRAP  = 1000

data_path    = "/infodev1/infodtao/AHA/Yue_UDP_data/data_table/"
lab_file     = data_path + "AHA_Baseline_characteristics_lab.csv"
smoking_file = data_path + "AHA_Baseline_characteristics_smoking.csv"
ischemic_tte = data_path + "ischemic_tte_label_pw_365.csv"
bleeding_tte = data_path + "bleeding_tte_label_pw_365.csv"

## 2. Load & Merge Data

In [2]:
lab_df      = pd.read_csv(lab_file)
smoking_df  = pd.read_csv(smoking_file)[['PATIENT_CLINIC_NUMBER', 'smoke_label']]
ischemic_df = pd.read_csv(ischemic_tte)
bleeding_df = pd.read_csv(bleeding_tte)

# Merge lab + smoking first
base_df = pd.merge(lab_df, smoking_df, on='PATIENT_CLINIC_NUMBER', how='inner')

# Select needed columns
base_df = base_df[['PATIENT_CLINIC_NUMBER', 'age', 'smoke_label', 'diabetes_label',
                    'acute_ischemic_heart_at_presentation', 'history_of_ischemic_heart',
                    'heart_failure_label']]

# Merge with ischemic outcomes
df = pd.merge(base_df, ischemic_df,
              left_on='PATIENT_CLINIC_NUMBER', right_on='patient_id', how='inner'
).drop(columns=['patient_id'])

# Merge with bleeding outcomes
df = pd.merge(df, bleeding_df,
              left_on='PATIENT_CLINIC_NUMBER', right_on='patient_id', how='inner',
              suffixes=('_ischemic', '_bleeding')
).drop(columns=['patient_id', 'PATIENT_CLINIC_NUMBER'])

print(f"Cohort size: {df.shape[0]} patients")

Cohort size: 29032 patients


## 3. Compute Partial DAPT Score (7 of 9 Variables)

In [3]:
# Age points: <65=0, 65-74=-1, >=75=-2
df['age_points'] = np.where(df['age'] < 65,  0,
                   np.where(df['age'] < 75, -1, -2))

# CHF/heart failure = 2 points
df['hf_points']  = df['heart_failure_label'] * 2

# Partial DAPT score — 7 of 9 variables
df['dapt_score'] = (
      df['age_points']
    + df['smoke_label']                           # +1 cigarette smoking
    + df['diabetes_label']                        # +1 diabetes mellitus
    + df['acute_ischemic_heart_at_presentation']  # +1 MI at presentation
    + df['history_of_ischemic_heart']             # +1 prior PCI or MI (proxy)
    + df['hf_points']                             # +2 CHF or LVEF<30%
    # stent diameter <3mm  (+1 pt) -- NOT AVAILABLE in structured EHR
    # vein graft stent     (+2 pts) -- NOT AVAILABLE in structured EHR
)

print(f"Score range  : {df['dapt_score'].min()} to {df['dapt_score'].max()}")

Score range  : -2 to 6


## 4. Train / Test Split - C-index with 95% Bootstrap CI

In [4]:
df_clean = df.dropna(subset=['dapt_score', 'time_to_event_ischemic', 'ischemic_label', 
                              'time_to_event_bleeding', 'bleeding_label'])

_, test_df = train_test_split(df_clean, test_size=0.2, random_state=RANDOM_STATE,
                               stratify=df_clean['ischemic_label'])

scores = -test_df['dapt_score'].values

# Ischemic C-index
times_i  = test_df['time_to_event_ischemic'].values
events_i = test_df['ischemic_label'].values
c_index_i = concordance_index(times_i, scores, events_i)

# Bleeding C-index
times_b  = test_df['time_to_event_bleeding'].values
events_b = test_df['bleeding_label'].values
c_index_b = concordance_index(times_b, scores, events_b)

# Bootstrap 95% CI for both
rng  = np.random.default_rng(RANDOM_STATE)
n    = len(scores)
boot_i, boot_b = [], []
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n, n)
    try:
        boot_i.append(concordance_index(times_i[idx], scores[idx], events_i[idx]))
        boot_b.append(concordance_index(times_b[idx], scores[idx], events_b[idx]))
    except Exception:
        pass

lo_i, hi_i = np.percentile(boot_i, [2.5, 97.5])
lo_b, hi_b = np.percentile(boot_b, [2.5, 97.5])

print("=" * 60)
print(f"DAPT Score (7/9 vars) — Ischemic C-index : {c_index_i:.3f} (95% CI: {lo_i:.3f}–{hi_i:.3f})")
print(f"DAPT Score (7/9 vars) — Bleeding C-index : {c_index_b:.3f} (95% CI: {lo_b:.3f}–{hi_b:.3f})")
print(f"N bootstrap resamples                    : {len(boot_i)}")
print("=" * 60)
print()

DAPT Score (7/9 vars) — Ischemic C-index : 0.580 (95% CI: 0.568–0.594)
DAPT Score (7/9 vars) — Bleeding C-index : 0.579 (95% CI: 0.557–0.599)
N bootstrap resamples                    : 1000

